# N138 · Sparse Table与静态RMQ

**目标：** 利用幂区间预处理回答静态幂等查询。

**先修：** N136, N056。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 稀疏表；log表；min/max/gcd幂等；重叠块；不可动态更新；Disjoint Sparse Table拓展。

## 从问题到算法

静态RMQ没有更新时，可以预存所有2^k长度块。查询长度L取两个长度2^floor(log2 L)块覆盖区间，允许重叠，因为min/max满足幂等性：重复合并同一元素不改变结果。求和不满足这个性质，不能照搬两块重叠公式。

## 最小实现

**本章接口：** `SparseTableMin`、`SparseTableMax`、`static_window_max(nums, k)`

In [1]:
class _SparseTable:
    def __init__(self,values,combine):
        self.n=len(values); self.combine=combine; self.table=[list(values)]; width=2
        while width<=self.n:
            half=width//2; old=self.table[-1]
            self.table.append([combine(old[i],old[i+half]) for i in range(self.n-width+1)]); width*=2
    def query(self,left,right):
        if not 0<=left<right<=self.n: raise IndexError('nonempty half-open interval required')
        k=(right-left).bit_length()-1; width=1<<k
        return self.combine(self.table[k][left],self.table[k][right-width])

class SparseTableMin(_SparseTable):
    def __init__(self,values): super().__init__(values,min)

class SparseTableMax(_SparseTable):
    def __init__(self,values): super().__init__(values,max)

def static_window_max(nums,k):
    if k<=0: raise ValueError('positive window length required')
    tree=SparseTableMax(nums)
    return [tree.query(i,i+k) for i in range(len(nums)-k+1)]

## 正确性、前提与复杂度

两个选定块的并集恰好覆盖查询区间，交集可能非空。结合律与幂等性保证重复元素不影响min/max，因此答案正确。若需要一般结合运算，可用不相交分块或Disjoint Sparse Table，不能仅修改combine就声称本算法通用。

**代价：** 预处理O(n log n)时间空间，查询O(1)次整数索引与聚合。数组静态；更新后原表失效。固定窗口一次性最大值通常用单调队列O(n)更省，不因学过稀疏表就强行使用。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[4,2,5,1,7]; st=SparseTableMin(a)
show_table(['块长','各起点块最小值'],[(1<<k,row) for k,row in enumerate(st.table)])
print('长度5由[0,4)与[1,5)两块覆盖：',st.query(0,5))

块长,各起点块最小值
1,"[4, 2, 5, 1, 7]"
2,"[2, 2, 1, 1]"
4,"[1, 1]"


长度5由[0,4)与[1,5)两块覆盖： 1


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert SparseTableMin([4,2,5,1]).query(0,4)==1
assert static_window_max([1,3,-1,-3,5,3,6,7],3)==[3,3,5,5,6,7]
from random import Random
rng=Random(138)
for n in range(1,35):
    a=[rng.randrange(-9,10) for _ in range(n)]; small=SparseTableMin(a); big=SparseTableMax(a)
    for l in range(n):
        for r in range(l+1,n+1): assert small.query(l,r)==min(a[l:r]) and big.query(l,r)==max(a[l:r])
assert static_window_max([1,2],3)==[]
# 对和使用重叠公式会重复计入中间元素。
a=[1,2,3]; assert sum(a[:2])+sum(a[1:])!=sum(a)
print('N138: 所有本章断言通过')

N138: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 构造用重叠两块求sum的错误反例。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 比较RMQ与单调队列解滑窗。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 239. Sliding Window Maximum（extension）
- 1438. Longest Continuous Subarray With Absolute Diff Less Than or Equal to Limit（extension）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。